# Type 2 Diabetes Mellitus: Global Prevalence and Risk Factors

## **Project Overview**
This project explores the global prevalence of Type 2 Diabetes (T2D) through exploratory data analysis.

## **Introduction**
In this project, we will examine global prevalence patterns of Type 2 Diabetes (T2D) and its associated risk factors. T2D is one of the fastest-growing diseases worldwide. It is estimated that among adults 20-79, T2D had increased by 74 million from 463 in 2019 (9% of population according to 9th edition of the IDF reporting) to 537 million (10.5% population) in 2021. And it is projected to affect 783 million adults age 20-79 by 2045$^1$. This chronic condition impacts multiple organs, including the nerves, heart, kidneys, blood vessels, and eyes, significantly reducing quality of life and life expectancy$^2$. Consequently, T2D places a substantial burden on healthcare systems globally. Diabetes caused at least USD 1 trillion dollars in health expenditure – a 338% increase over the last 17 years$^3$ and it should be the primary concern for the regions with high disease burden.
Hence, the goal of this project is to:
1. Analyse prevalence patterns of T2D worldwide to identify countries with the highest rates where interventions are most needed.
2. Investigate key risk factors contributing to T2D prevalence, that are most worth attention to reduce the prevalence of T2D, including:
    - Age, Low physical activity, High BMI$^4$.
    - Consumption of sugar-sweetened beverages$^5$.
    - Tabacco use$^6$.
---
## **Data Source**
For this project, we used T2D, low physical activity, high BMI, consumption of SSBs, tabaco use statistics for 204 countries and locations, that are publicly accessible with Global Health Data Exchange's (GHDx) visualisation tools$^7$, provided by the Institute for Health Metrics and Evaluation (IHME). GHDx offers a wide range of health-related datasets, including survey data, scientific study results, and statistically modeled estimates$^8$. These values availabale on this resource are estimates

We used T2D count values for the various age groups (<5, 5-9,  10-14, 15-19, 20-24, 25-29, 30-34, 35-39, 40-44, 45-49, 50-54, 55-59, 60-64, 65-69, 70-74, 75-79, 80-84, 85-89, 90-94, 95>) in 2023. We also used the population estimates in all those countries and age groups in 2023. Furthermore, we used age-standardised T2D, low physical activity, high BMI, tabaco usage, and high consumption of SSBs prevalence percentages spanning from 1990-2023.

## **Methods**  
- Exploratory Analysis: Age as a Risk Factor for Type 2 Diabetes Mellitus
- Exploratory Analysis: Age-Standardisation and Global Comparison
  Choropleth: Visualizing Global Prevalence
  Scatter ploth: T2D values Man vs Females
  Line ploth: Regions
- Risk Factors of Type 2 Diabetes
  Multiple Regression Analysis

---
## **References**  
1. https://doi.org/10.1016/j.diabres.2021.109119.
2. https://link.springer.com/article/10.1186/s12916-025-03890-w.
3. IDF Diabetes Atlas 11th Edition, https://diabetesatlas.org/.
4. https://www.niddk.nih.gov/health-information/diabetes/overview/risk-factors-type-2-diabetes.
5. https://www.nature.com/articles/s41591-024-03345-4.
6. https://www.who.int/news-room/fact-sheets/detail/diabetes
7. https://vizhub.healthdata.org/gbd-results/.
8. https://ghdx.healthdata.org/




## Libraries and Data Import and Wrangling
### Libraries

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import json
import numpy as np
import geopandas as gpd
from matplotlib.ticker import MaxNLocator

### Data Import and Wrangling

In [ ]:
# Upload T2D counts in 2023 for all countries for various age groups
df_T2D = pd.read_csv("../data/prev_num_allc_2023_both_T2D.csv")
# Upload the population counts in 2023 for all countries for various age groups
df_population = pd.read_csv("../data/pop_allc_allc_2023.csv")
df_population = df_population[["location_name", "val", "sex_name", "age_name"]]
# Upload age standardized percentages of T2D for all countries between 1990-2023
df_agestand = pd.read_csv("../data/agestand_allc_ally_perc_T2D.csv")
# Upload summary exposure values for low physical activity,
# high BBI, high consumption of SSB
df_sev = pd.read_csv("../data/SEV_allc_ally_BBI_SSB_lowact.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/prev_num_allc_2023_both_T2D.csv'

In [ ]:
df_T2D.head(2)

In [ ]:
df_population.head(2)

In [ ]:
df_agestand.head(2)

In [ ]:
df_sev.head(2)

In [ ]:
# The three columns in df_agestand, val, upper, and lower, are proportions but
# we need percentages for further plotting.
# Hence, convert the three columns with proportions to percentage.
cols_to_fix = ['val', 'upper', 'lower']
df_agestand[cols_to_fix] = df_agestand[cols_to_fix] * 100

In [ ]:
# Concatenate both dfs for further wrangling
df_concat = pd.concat([df_T2D, df_agestand])

In [ ]:
# Merge the two datasets to get the population numbers for each age group for each country, except
# age-standardized group
df_merged = df_concat.merge(df_population[
    ["location_name", "val", "sex_name", "age_name"]], on=["location_name", "sex_name", "age_name"],
                            how="left", suffixes=["_T2D", "_population"])

In [ ]:
df = df_merged.copy()

Refer to the official documentation ([see official documentation](https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html)) on how to use pandas.Categorical() function to categorise pandas dataframe columns.


In [ ]:
# Set the age_name column into categories for further plotting
# Define the order list in the same sequence as it is required for plotting, except age-standardized
age_order = [
    "<5 years", "5-9 years", "10-14 years", "15-19 years",
    "20-24 years", "25-29 years", "30-34 years", "35-39 years",
    "40-44 years", "45-49 years", "50-54 years", "55-59 years",
    "60-64 years", "65-69 years", "70-74 years", "75-79 years",
    "80-84 years", "85-89 years", "90-94 years", "95+ years", "Age-standardized"
]
# Categorise the age_name column
df["age_name"] = pd.Categorical(df["age_name"], categories=age_order, ordered=True)
df = df.sort_values("age_name")

It is also necessary to create a region column indicating the geographical region to which each country belongs. The mapping of countries to regions was manually curated based on the [GHDx country classifications](https://ghdx.healthdata.org/countries). Subsequently, certain regions were aggregated into larger super-regions to reduce the total number of regions, thereby improving clarity and interpretability in subsequent plots.

In [ ]:
# Upload  the manually curate .json file that contains the region information
with open("../data/regions_updated.json", "r", encoding="utf-8") as f:
    regions = json.load(f)

country_to_region = {
    country: region
    for region, countries in regions.items()
    for country in countries
}
# Add an extra region "Global", that includes all countries,
# to the region list for further plotting
country_to_region["Global"] = "Global"

df["region"] = [country_to_region[acountry] for acountry in df["location_name"].to_list()]

In [ ]:
# Check out all the regions
df["region"].unique()

In [ ]:
df.tail(2)

## Exploratory Analysis: Age as a Risk Factor for Type 2 Diabetes Mellitus

In this section, we will analyse T2D prevalence across different age groups for 2023 in a line plot. We will plot T2D prevalence percentage for different geophraphical regions through different age groups. However, we firstly need to derive the prevalence percentages and its 95% confidence interval (CI) values using the T2D counts and the population sizes that are present in the above uploaded datasets. We will derive these values using the following eight steps:

**Step 1: Filter the Dataframe**  
Filter the dataset to retain only numeric prevalence values and exclude pre-existing age-standardised values. We also will remove unused categories. Refer to official documentation ([see official documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.html)) on how to pandas.Series.cat.remove_unused_categories() to remove unused categories.

**Step 2: Calculate Standard Error (SE) and Variance**  
Derive the SE for each country using the 95% CI lower and upper bounds. This is based on the formula:  
$SE = \frac{95\%Upper - 95\%Lower}{1.96 \times 2}$  
Refer to [PennState Statistics: Confidence Intervals](https://online.stat.psu.edu/statprogram/reviews/statistical-concepts/confidence-intervals) on how to derive SE from the confidence interval values and how to calculate variance.  


**Step 3: Pool Countries into Regions**  
Aggregate individual country data into their respective geographical regions.

**Step 4: Calculate Regional SE and Confidence Intervals**  
Calculate the pooled SE and 95% CI for the geographical regions. Refer to official documentation [see official documentation](https://numpy.org/doc/2.2/reference/generated/numpy.sqrt.html) on how to use numpy.sqrt() function to get the square root.

**Step 5: Calculate Proportion and Percentage**  
Determine the proportion of the population affected, followed by the conversion to percentage values.

**Step 6: Calculate SE for the Proportion**  
Refer to [Statistichowto](https://www.statisticshowto.com/binomial-approximation/) on how to calculate the SE of the proportion.

**Step 7: Calculate 95% CI for the Percentages**  
Establish the final confidence margins for the calculated regional percentages.

**Step 8: Add Age Midpoint Column (`age_mid`)**  
Create a numeric `age_mid` column representing the average midpoint of each age group. This step is necessary to plot age on the x-axis in the following plot.

---

In [ ]:
# Step 1: Filter the df
df_number = df.query("sex_name == 'Both' and metric_name == 'Number' and age_name != 'Age-standardized'")
# Remove unused categories in age_name column (Age-standardized is absent must be removed)
df_number.loc[:,"age_name"] = df_number["age_name"].cat.remove_unused_categories()
df_number.head(2)

In [ ]:
# Step 2: Calculate SE and variance of the countries using the 95% CI lower and higher values
df_number["se"] = (df_number["upper"] - df_number["lower"]) / (1.96 * 2)
df_number["var"] = df_number["se"] ** 2

In [ ]:
# Step 3: Aggregate by region and age_name and sum T2D values, population, and variances
# to get these values of the regions
df_region = (
    df_number.groupby(["region", "age_name"], as_index=False)
    .agg(
        val_T2D=("val_T2D", "sum"),
        val_population=("val_population", "sum"),
        var_T2D=("var", "sum")
    )
)

In [ ]:
# Step 4: Calculate SE, and 95% CI values for the geographical regions
df_region["se_Number"] = np.sqrt(df_region["var_T2D"])
df_region["lower_Number"] = df_region["val_T2D"] - 1.96 * df_region["se_Number"]
df_region["upper_Number"] = df_region["val_T2D"] + 1.96 * df_region["se_Number"]

In [ ]:
# Step 5: Calculate the proportion followed by percentage
df_region["Proportion"] = df_region["val_T2D"] / df_region["val_population"]
df_region["Proportion"] = df_region["Proportion"].fillna(0)
df_region["Percent"] = df_region["Proportion"] * 100

In [ ]:
# Step 6: Calculate standard error for the proportion
df_region["se_Proportion"] = np.sqrt(
    df_region["Proportion"] * (1 - df_region["Proportion"]) / df_region["val_population"]
)
df_region["se_Proportion"] = df_region["se_Proportion"].fillna(0)

In [ ]:
# Step 7: Calculate 95% CI for the percent
df_region["lower_Percent"] = (df_region["Proportion"] - 1.96 * df_region["se_Proportion"]) * 100
df_region["upper_Percent"] = (df_region["Proportion"] + 1.96 * df_region["se_Proportion"]) * 100

In [ ]:
# Step 8: Add the age midpoint for plotting
def age_midpoint(age):
    if age.startswith("<"):
        return 2.5
    if "+" in age:
        return 97.5
    start, end = age.split("-")
    return (int(start) + int(end.split()[0])) / 2

df_region["age_mid"] = df_region["age_name"].apply(age_midpoint)


df_region.head()

In [ ]:
# Create the list with unique regions and set grid dimensions
regions = df_region["region"].unique()
n = len(regions)
nrows = 3
ncols = 4

# Identify which subplot indices belong to the bottom row for X-axis labeling
label_xaxis = list(range((nrows * ncols - ncols), nrows * ncols))

# Initialize the figure with shared axes for better comparison
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * 3, nrows * 3),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for count, (ax, region) in enumerate(zip(axes, regions)):
    # Filter data for the specific region
    region_df = df_region[df_region["region"] == region]

    # Plot the prevalence trend line
    ax.plot(
        region_df["age_mid"],
        region_df["Percent"],
        label="Mean",
        marker="o",
        markeredgewidth=0
    )

    # Add shaded area for the 95% Confidence Interval
    ax.fill_between(
        region_df["age_mid"],
        region_df["lower_Percent"],
        region_df["upper_Percent"],
        alpha=0.2
    )

    # Set Y-axis labels only for the leftmost column to reduce clutter
    col = count % ncols
    if col == 0:
        ax.set_ylabel("Prevalence (%)")
    else:
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelleft=False)

    # Set X-axis labels only for the bottom row
    if count in label_xaxis:
        ax.set_xlabel("Age (years)")
    else:
        ax.set_xlabel("")
        ax.tick_params(axis="x", labelbottom=False)

    # Optimize tick density and style
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.set_title(region, fontsize=8)

    # Clean up chart aesthetics
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide any unused subplot tiles in the grid
for ax in axes[len(regions):]:
    ax.axis("off")

fig.suptitle(
    "Type 2 Diabetes Prevalence for Different Age Groups in Different World Regions",
    fontsize=16,
    y=1.04
)

plt.tight_layout()
plt.show()

## Exploratory Analysis: Age-Standardisation and Global Comparison

In the previous section, we identified age as a major risk factor for T2D, with older age groups showing higher prevalence. To enable fair comparisons between countries with different age distributions, we will now use an age-standardised dataset. This adjustment removes age as a confounding factor, allowing accurate region-to-region and country-to-country comparisons.

### Choropleth: Visualising Global Prevalence

In this section, we will plot the percentage of T2D prevalence in 2023 on a choropleth map, which displays values across geographic regions. A choropleth provides a visual representation of global patterns. Refer to [towardsdatascience - a beginners guide to create a choropleth map](https://towardsdatascience.com/a-beginners-guide-to-create-a-cloropleth-map-in-python-using-geopandas-and-matplotlib-9cc4175ab630/) on how to create a choropleth map in python using geopandas and matplotlib. In addition, refer to GeoPandas official documentation ([see official documentation](https://geopandas.org/en/stable/gallery/choropleths.html)) that enables to create choropleths.


In [ ]:
# Define the URL that contains the info with the countries' cordinates to
url = "https://r2.datahub.io/clvyjaryy0000la0cxieg4o8o/main/raw/data/countries.geojson"
# Load in the cordinates as the GeoPandas object
df_map = gpd.read_file(url)
# Remove Antarctica from the df because it is not necessary for plotting
df_map = df_map[df_map["name"] != "Antarctica"]
# Check the GeoDataframe
df_map.head(2)

In [ ]:
# Define the plot's dimensions
plt.rcParams['figure.figsize'] = [50, 70] #height, width
df_map.plot()

In [ ]:
# Select the percentage values of T2D in 2023 that includes both genders
df_both_2023 = df.query("sex_name == 'Both' and year == 2023 and metric_name == 'Percent'")

The names of the countries in the df_map and df_both_2023 differ due to different countries' naming conventions. Hence, we have to identify which countries are different among these countries and create a new column in df_map with the same country name as in the df_both_2023. We will do this in the following steps:

- **Step 1: Identify countries with mismathing names**
- **Step 2: Manually curate a dictionary with mismatching country names**
- **Step 3: Add a new column in df_map with country names**

In [ ]:
# Step 1: Identify countries with mismathing names
absent = []
for country in df_map.name:
    if country not in df_both_2023.location_name.unique():
        absent.append(country)
df_absent = pd.DataFrame(absent)
df_absent

In [ ]:
# Step 2: Manually curate a dictionary with mismatching country names
country_map = {
    "Bolivia": "Bolivia (Plurinational State of)",
    "Dhekelia Sovereign Base Area": "Dhekelia Sovereign Base Area",
    "Syria": "Syrian Arab Republic",
    "Somaliland": "Somalia",
    "South Korea": "Republic of Korea",
    "North Korea": "Democratic People's Republic of Korea",
    "Western Sahara": "Western Sahara",
    "Republic of the Congo": "Democratic Republic of the Congo",
    "Saint Martin": "Saint Martin",
    "Sint Maarten": "Sint Maarten",
    "Russia": "Russian Federation",
    "Vietnam": "Viet Nam",
    "Kosovo": "Kosovo",
    "Turkey": "Türkiye",
    "Laos": "Lao P`eople's Democratic Republic",
    "Iran": "Iran (Islamic Republic of)",
    "Liechtenstein": "Liechtenstein",
    "Ivory Coast": "Côte d'Ivoire",
    "Republic of Serbia": "Serbia",
    "East Timor": "Timor-Leste",
    "Brunei": "Brunei Darussalam",
    "eSwatini": "Eswatini",
    "US Naval Base Guantanamo Bay": "US Naval Base Guantanamo Bay",
    "Brazilian Island": "Brazilian Island",
    "Moldova": "Republic of Moldova",
    "Gibraltar": "Gibraltar",
    "Venezuela": "Venezuela (Bolivarian Republic of)",
    "Hong Kong S.A.R.": "Hong Kong S.A.R.",
    "Vatican": "Vatican City",
    "Northern Cyprus": "Northern Cyprus",
    "Cyprus No Mans Area": "Cyprus No Mans Area",
    "Siachen Glacier": "Siachen Glacier",
    "Baykonur Cosmodrome": "Baykonur Cosmodrome",
    "Akrotiri Sovereign Base Area": "Akrotiri Sovereign Base Area",
    "Southern Patagonian Ice Field": "Southern Patagonian Ice Field",
    "Bir Tawil": "Bir Tawil",
    "New Caledonia": "New Caledonia",
    "Curaçao": "Curaçao",
    "Aruba": "Aruba",
    "The Bahamas": "Bahamas",
    "Turks and Caicos Islands": "Turks and Caicos Islands",
    "Saint Pierre and Miquelon": "Saint Pierre and Miquelon",
    "Pitcairn Islands": "Pitcairn Islands",
    "French Polynesia": "French Polynesia",
    "French Southern and Antarctic Lands": "French Southern and Antarctic Lands",
    "United States Minor Outlying Islands": "United States Minor Outlying Islands",
    "Montserrat": "Montserrat",
    "Saint Barthelemy": "Saint Barthelemy",
    "Anguilla": "Anguilla",
    "British Virgin Islands": "British Virgin Islands",
    "Cayman Islands": "Cayman Islands",
    "Heard Island and McDonald Islands": "Heard Island and McDonald Islands",
    "Saint Helena": "Saint Helena",
    "São Tomé and Principe": "Sao Tome and Principe",
    "Jersey": "Jersey",
    "Guernsey": "Guernsey",
    "Isle of Man": "Isle of Man",
    "Aland": "Aland",
    "Faroe Islands": "Faroe Islands",
    "Indian Ocean Territories": "Indian Ocean Territories",
    "British Indian Ocean Territory": "British Indian Ocean Territory",
    "Norfolk Island": "Norfolk Island",
    "Wallis and Futuna": "Wallis and Futuna",
    "Federated States of Micronesia": "Micronesia (Federated States of)",
    "South Georgia and the Islands": "South Georgia and the Islands",
    "Falkland Islands": "Falkland Islands",
    "Coral Sea Islands": "Coral Sea Islands",
    "Spratly Islands": "Spratly Islands",
    "Clipperton Island": "Clipperton Island",
    "Macao S.A.R": "Macao S.A.R",
    "Ashmore and Cartier Islands": "Ashmore and Cartier Islands",
    "Bajo Nuevo Bank (Petrel Is.)": "Bajo Nuevo Bank (Petrel Is.)",
    "Serranilla Bank": "Serranilla Bank",
    "Scarborough Reef": "Scarborough Reef"}

In [ ]:
# Step 3: Add a new column in df_map with country names
df_map["country_name_GBD"] = [
    acountry if acountry in df_both_2023.location_name.unique()
    else country_map[acountry]
    for acountry in df_map.name
]

# Merge df_map and df_both_2023 on a new common column with the country names
merged = df_map.merge(df_both_2023, left_on='country_name_GBD', right_on='location_name', how='inner')
merged.head()

In [ ]:

data = merged.groupby(["country_name_GBD", "geometry", "region"]).mean("val").reset_index()
data = gpd.GeoDataFrame(data)
data.head()

In [ ]:
# Define the variable to plot
variable = 'val_T2D'
# Define the min and max values that will appear on the plot
vmin, vmax = 0, round(data["val_T2D"].max(), 1)
fig, ax = plt.subplots(1, figsize=(30, 10))
ax.axis('off')
ax.set_title('Diabetes Prevalence by Country in 2023',
             fontdict={'fontsize': '25', 'fontweight' : '3'})
# Add an annotation for the color scale
ax.annotate('Prevalence (%)', xy=(0.64, .04), xycoords='figure fraction',
            fontsize=12, color='#555555')

# Plot the data on the map
data.plot(column=variable, cmap='Blues', linewidth=0.8, ax=ax, edgecolor='0.8',
            vmin=vmin, vmax=vmax)

# Create a colorbar for the map
sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
fig.colorbar(sm, ax=ax)

plt.show()

### Scatter ploth: T2D values Man vs Females

In this section, we will plot the percentages of T2D prevalence for males vs females across different countries in 2023 on a scatter plot. Each point represents a country, and the points are color-coded based on the region to which the country belongs. This visualization helps reveal gender differences in T2D prevalence and regional patterns at a glance.

In [ ]:
df_gender = df.query("sex_name in ['Male', 'Female'] and metric_name == 'Percent'")
df_gender.head(2)

In [ ]:
df_pivot = (
    df_gender
    .pivot(
        index=["location_name", "year", "region"],
        columns="sex_name",
        values="val_T2D"
    )
    .reset_index()
)

# Select only the year 2023
df_pivot = df_pivot[df_pivot["year"] == 2023]
df_pivot.head(2)

In [ ]:
# See the list of colors https://en.wikipedia.org/wiki/List_of_colors:_A%E2%80%93F
colors_11 = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#e377c2",  # pink
    "#000000",  # black
    "#17becf",  # cyan
    "#FF00FF",  # fuchsia
    "#80FF00",  # chartreuse (web)
    "#00CC99",  # caribbean green
    "#964B00",  # brown
]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

x = df_pivot["Male"]
y = df_pivot["Female"]
regions = df_pivot["region"].astype("category")
colors = regions.cat.codes

regions_unique = df_pivot["region"].unique()

for i, region in enumerate(regions_unique):
    df_temp = df_pivot[df_pivot["region"] == region]
    x = df_temp["Male"]
    y = df_temp["Female"]
    ax.scatter(
        x,
        y,
        alpha=1,
        edgecolors="black",
        linewidths=0.3,
        label=region,
        c=colors_11[i]
    )
ax.legend(title="Region")
plt.xlabel("Men")
plt.ylabel("Women")
plt.xticks([0, 5, 10, 15, 20, 25])
plt.yticks([0, 5, 10, 15, 20, 25])
plt.title("Diabetes Prevalence Between Age-standardised Men and Women")

## Plot 4

In [ ]:
from matplotlib.collections import LineCollection

In [ ]:
df_agestand = df[df["metric_name"] == "Percent"]

In [ ]:
for gender in ["Male", "Female"]:

    color = {
        "Male": "lightskyblue",
        "Female": "red"
    }
    color = color[gender]


    df_agestand_gender = df_agestand[df_agestand["sex_name"] == gender]
    regions = df_agestand_gender['region'].unique()
    n = len(regions)
    nrows=3
    ncols=4

    label_xaxis = list(range((nrows * ncols - ncols), nrows * ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*3, nrows*3), sharex=True, sharey=True)

    axes = axes.flatten()

    count = 0

    for ax, region in zip(axes, regions):

        if region == "Global":
            region_df = df_agestand_gender
        else:
            region_df = df_agestand_gender[df_agestand_gender['region'] == region]

        for location in region_df["location_name"]:

            df_location = region_df[region_df["location_name"] == location]
            ax.plot(df_location['year'], df_location["val_T2D"], linewidth=0.1, color=color, alpha=0.2)


        col = count % ncols
        if col == 0:
            ax.set_ylabel("Prevalence (%)")
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)


        if count in label_xaxis:
            ax.set_xlabel("Age (years)")
        else:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelleft=False)
        count += 1


        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.set_title(region, fontsize=8)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)


    plt.xlabel("Age group")
    plt.tight_layout()
    fig.suptitle(
        "Type 2 Diabetes Prevalence",
        y=1.04,
        fontsize=16)
    plt.show()

### What Drives T2D?

In [ ]:
df.head()

In [ ]:

sev.head()

In [ ]:
df_sev = sev.pivot(
    index=["location_name", "year"],
    columns="rei_name",
    values=["val", ]
).reset_index()

df_sev.columns = df_sev.columns.droplevel(0)
df_sev.head()
df_sev.columns = ["location_name", "year", "high_SSB", "high_BMI", "low_physical_activity"]

In [ ]:
df_percent_both = df.query("metric_name == 'Percent' and sex_name == 'Both'")


In [ ]:
cols_to_keep = ["location_name", "year", "val_T2D"]

In [ ]:
df_merged = df_percent_both[cols_to_keep].merge(df_sev, on=["location_name", "year"])
df_merged.head()

In [ ]:

df_indexed = df_merged.set_index(["location_name", "year"])
df_indexed.head()


Explanation on how to use linearmodels for multiple linear regression:
https://www.youtube.com/watch?v=TYTvLgi4mVc

Linearmodels documentation
https://bashtage.github.io/linearmodels/panel/panel/linearmodels.panel.model.PanelOLS.html

In [ ]:
from linearmodels.panel import PanelOLS



y = df_indexed["val_T2D"]
X = df_indexed[["high_SSB", "high_BMI", "low_physical_activity"]]

mod = PanelOLS.from_formula("val_T2D ~ 1 + high_SSB + high_BMI + low_physical_activity", df_indexed)
res = mod.fit()
res.summary
# model = PanelOLS(
#     y,
#     X,
#     entity_effects=True,  # country
#     time_effects=True     # year
# )

# results = model.fit(cov_type="clustered", cluster_entity=True)
# print(results)